# DeepFM — A Factorization-Machine based Neural Network for CTR Prediction (2017)

_Papers · Recommender Systems_

**One shared embedding table feeds a Factorization Machine (low-order) and a deep network (high-order); add their scores, then sigmoid.**

---

This notebook is a practice scaffold for the **DeepFM — A Factorization-Machine based Neural Network for CTR Prediction (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score

torch.manual_seed(0); np.random.seed(0)

# --- 0. Sanity-check the worked example: FM order-2 == <e1,e2>, two ways. ---
e1 = torch.tensor([0.2, -0.1, 0.4]); e2 = torch.tensor([0.5, 0.3, -0.2])
direct = (e1 * e2).sum()                                  # inner product
s = e1 + e2
identity = 0.5 * ((s * s).sum() - (e1 * e1).sum() - (e2 * e2).sum())
print("worked example:  <e1,e2> =", round(direct.item(), 4),
      " square-of-sum identity =", round(identity.item(), 4))   # both -0.01


# --- 1. Tiny synthetic CTR data with MIXED interaction orders. ---
# CTR = click-through rate. Each field is a categorical feature (one value per row).
N = 10000
field_cards = [40, 40, 4, 4, 4]            # 2 high-card "pair" fields + 3 low-card "parity" fields
n_fields = len(field_cards)
g = torch.Generator().manual_seed(3)
idx = torch.stack([torch.randint(0, c, (N,), generator=g) for c in field_cards], dim=1)  # (N, 5)

# Low-order (order-2): a low-rank id-id interaction <U[a], V[b]> -> exactly FM's model.
r = 4
U = torch.randn(40, r, generator=g); Vp = torch.randn(40, r, generator=g)
pair = (U[idx[:, 0]] * Vp[idx[:, 1]]).sum(1) * 0.8
# High-order (order-3): a three-way parity -> needs DNN depth, FM order-2 cannot reach it.
b2 = (idx[:, 2] < 2).long(); b3 = (idx[:, 3] < 2).long(); b4 = (idx[:, 4] < 2).long()
triple = 1.6 * ((b2 ^ b3 ^ b4).float() * 2 - 1)
logit = pair + triple + 0.1 * torch.randn(N, generator=g)
y = torch.bernoulli(torch.sigmoid(logit), generator=g)     # click / no-click labels

offsets = torch.tensor(np.cumsum([0] + field_cards[:-1]), dtype=torch.long)
feat_idx = idx + offsets                                    # global feature index per field
total_feats = sum(field_cards)
ntr = 8000; tr = slice(0, ntr); te = slice(ntr, N)
k = 8                                                       # embedding size (FM latent size)


# --- 2. DeepFM (built by hand). mode in {"fm","dnn","deepfm"} is the ablation switch. ---
class DeepFM(nn.Module):
    def __init__(self, mode="deepfm"):
        super().__init__()
        self.mode = mode
        self.w0 = nn.Parameter(torch.zeros(1))             # global bias
        self.w1 = nn.Embedding(total_feats, 1)             # order-1 weights <w,x>
        self.V  = nn.Embedding(total_feats, k)             # SHARED embeddings (V_i == e_i)
        nn.init.normal_(self.w1.weight, std=0.01)
        nn.init.normal_(self.V.weight,  std=0.05)
        H = n_fields * k
        self.mlp = nn.Sequential(nn.Linear(H, 64), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(32, 1))

    def forward(self, fi):
        emb = self.V(fi)                                   # (B, n_fields, k)  -- shared e_i
        # FM order-2 via the O(kd) identity: 0.5[(sum e)^2 - sum(e^2)], summed over k.
        s   = emb.sum(1)                                    # (B, k)
        fm2 = 0.5 * ((s * s) - (emb * emb).sum(1)).sum(1)  # (B,)
        fm1 = self.w1(fi).sum(1).squeeze(-1)               # order-1 <w,x>
        y_fm  = self.w0 + fm1 + fm2                         # FM scalar (Eqn. 2)
        a0    = emb.reshape(emb.size(0), -1)               # a^(0) = [e1..em]  -- SAME embeddings
        y_dnn = self.mlp(a0).squeeze(-1)                   # DNN scalar
        if self.mode == "fm":  return y_fm                 # ablation: FM only
        if self.mode == "dnn": return y_dnn                # ablation: DNN only
        return y_fm + y_dnn                                # Eqn. 1: sigmoid is in the loss


# --- 3. Train + evaluate (Adam, BCEWithLogitsLoss = one sigmoid inside the loss). ---
def run(mode, epochs=40, bs=256):
    torch.manual_seed(0)
    net = DeepFM(mode)
    opt = torch.optim.Adam(net.parameters(), lr=0.004, weight_decay=1e-5)
    lf  = nn.BCEWithLogitsLoss()
    Xtr, ytr = feat_idx[tr], y[tr]
    for ep in range(epochs):
        net.train(); perm = torch.randperm(ntr)
        for i in range(0, ntr, bs):
            b = perm[i:i + bs]
            opt.zero_grad(); loss = lf(net(Xtr[b]), ytr[b]); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        pte = torch.sigmoid(net(feat_idx[te]))             # apply sigmoid for scoring
        ll  = nn.functional.binary_cross_entropy(pte, y[te]).item()
        auc = roc_auc_score(y[te].numpy(), pte.numpy())
    return auc, ll

print("\nablation -- same data, same shared embeddings:")
for m in ["fm", "dnn", "deepfm"]:
    auc, ll = run(m)
    print(f"  {m:8s}  test AUC = {auc:.4f}   test LogLoss = {ll:.4f}")
# Our small run:  fm  AUC 0.6545 / LL 0.6761 ; dnn 0.8418 / 0.4946 ; deepfm 0.8585 / 0.4765.
# DeepFM beats FM-only AND DNN-only on both metrics -- the paper's effect on toy data.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_On a CTR task with both a pairwise and a three-way signal, does DeepFM (shared embeddings, FM + DNN) beat FM-only and DNN-only?_

In [ ]:
import torch, torch.nn as nn, numpy as np
from sklearn.metrics import roc_auc_score

# Reproduces the qualitative effect: DeepFM (shared FM + DNN) beats FM-only and DNN-only
# on a CTR task whose label mixes a pairwise signal and a three-way parity signal.
torch.manual_seed(0); np.random.seed(0)
N = 10000; field_cards = [40, 40, 4, 4, 4]; n_fields = len(field_cards); k = 8
g = torch.Generator().manual_seed(3)
idx = torch.stack([torch.randint(0, c, (N,), generator=g) for c in field_cards], dim=1)
r = 4
U = torch.randn(40, r, generator=g); Vp = torch.randn(40, r, generator=g)
pair = (U[idx[:, 0]] * Vp[idx[:, 1]]).sum(1) * 0.8                  # order-2 (FM)
b2 = (idx[:, 2] < 2).long(); b3 = (idx[:, 3] < 2).long(); b4 = (idx[:, 4] < 2).long()
triple = 1.6 * ((b2 ^ b3 ^ b4).float() * 2 - 1)                    # order-3 (needs depth)
y = torch.bernoulli(torch.sigmoid(pair + triple + 0.1 * torch.randn(N, generator=g)), generator=g)
offsets = torch.tensor(np.cumsum([0] + field_cards[:-1]), dtype=torch.long)
feat_idx = idx + offsets; total_feats = sum(field_cards); ntr = 8000

class DeepFM(nn.Module):
    def __init__(self, mode):
        super().__init__(); self.mode = mode
        self.w0 = nn.Parameter(torch.zeros(1)); self.w1 = nn.Embedding(total_feats, 1)
        self.V = nn.Embedding(total_feats, k)                       # SHARED embeddings
        nn.init.normal_(self.w1.weight, std=0.01); nn.init.normal_(self.V.weight, std=0.05)
        self.mlp = nn.Sequential(nn.Linear(n_fields * k, 64), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, fi):
        emb = self.V(fi); s = emb.sum(1)
        fm2 = 0.5 * ((s * s) - (emb * emb).sum(1)).sum(1)
        y_fm = self.w0 + self.w1(fi).sum(1).squeeze(-1) + fm2
        y_dnn = self.mlp(emb.reshape(emb.size(0), -1)).squeeze(-1)
        if self.mode == "fm":  return y_fm
        if self.mode == "dnn": return y_dnn
        return y_fm + y_dnn

def curve(mode, epochs=40, bs=256):
    torch.manual_seed(0)
    net = DeepFM(mode); opt = torch.optim.Adam(net.parameters(), lr=0.004, weight_decay=1e-5)
    lf = nn.BCEWithLogitsLoss(); out = []
    for ep in range(epochs):
        net.train(); perm = torch.randperm(ntr)
        for i in range(0, ntr, bs):
            b = perm[i:i + bs]; opt.zero_grad()
            loss = lf(net(feat_idx[b]), y[b]); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            p = torch.sigmoid(net(feat_idx[ntr:]))
            out.append(round(roc_auc_score(y[ntr:].numpy(), p.numpy()), 4))
    return out

for m in ["fm", "dnn", "deepfm"]:
    c = curve(m)
    print(m, [[ep, c[ep]] for ep in range(0, 40, 3)])
# DeepFM's AUC curve sits above both FM-only and DNN-only -- our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working DeepFM whose test AUC beats both single components. Flip the
            mode flag to run FM only and DNN only on the same data, then DeepFM.
            What ranking do you expect by AUC, and what does each gap tell you about the data's interaction
            orders?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run mode="fm": the model keeps only $y_{FM}$ (order-1 + order-2). — _FM captures the pairwise signal but has no mechanism for the three-way (order-3) parity, so its AUC caps low._
- Run mode="dnn": keep only $y_{DNN}$ (the deep MLP on the shared embeddings). — _Depth reaches the order-3 signal, lifting AUC well above FM &mdash; but the deep net handles the clean pairwise part less crisply._
- Run mode="deepfm": add both scores before the sigmoid. — _The FM component sharpens the low-order pairwise score while the deep part supplies the high-order signal &mdash; together they top both._

**Answer:** Expected ranking by AUC: DeepFM > DNN-only > FM-only. Our small run gave FM
                 0.6545, DNN 0.8418, DeepFM 0.8585 (LogLoss 0.6761 / 0.4946 / 0.4765 &mdash; DeepFM lowest too).
                 FM alone cannot reach the three-way parity, so it stalls; the deep net reaches it; DeepFM keeps
                 the deep net's high-order win and the FM's clean low-order score, so it edges out both.
                 This is the paper's qualitative claim reproduced on toy data &mdash; our small run, not the
                 paper's number.

</details>

**Problem 2.** A teammate builds DeepFM with two separate embedding tables &mdash; one for the FM branch,
            one for the deep branch &mdash; and reports it "works the same." What did they lose, and how would
            you spot it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the design break: the paper's contribution is one shared embedding feeding both halves (&sect;2.1.2). — _Shared embeddings mean the FM's low-order gradients and the deep part's high-order gradients jointly shape the same vectors &mdash; that coupling is the model._
- Note the cost: separate tables double the embedding parameters and decouple the two views of each feature. — _It degenerates into an ensemble of an independent FM and an independent DNN, summed &mdash; not DeepFM._
- Spot it: print the parameter that holds the embeddings; there should be exactly one nn.Embedding of shape (total_features, k) consumed by both branches. — _One table, two consumers, is the structural signature of DeepFM._

**Answer:** They lost the shared embedding &mdash; the paper's core idea. With two tables the model
                 is just an FM and a DNN trained side by side and summed; the embeddings no longer carry signal
                 jointly useful to low- and high-order terms, and parameter count doubles. Spot it by checking
                 that a single nn.Embedding tensor feeds both the FM identity and the MLP input.

</details>

**Problem 3.** In the worked example, $e_1=[0.2,-0.1,0.4]$ and $e_2=[0.5,0.3,-0.2]$ gave FM order-2 $=-0.01$.
            Suppose a third feature with embedding $e_3=[0,0,0]$ becomes active. How does the order-2 term
            change, and why is that the "safe" behavior?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the order-2 sums over all active pairs: now $(e_1,e_2),(e_1,e_3),(e_2,e_3)$. — _Adding a feature adds its pairings with every existing one._
- Compute the new pairs: $\langle e_1,e_3\rangle = 0$ and $\langle e_2,e_3\rangle = 0$ because $e_3$ is the zero vector. — _Any inner product with the zero vector is zero._
- Sum: $-0.01 + 0 + 0 = -0.01$ &mdash; unchanged. — _A zero embedding contributes nothing to the order-2 score._

**Answer:** The order-2 term stays $-0.01$: $e_3$'s two new inner products are both zero, so it adds
                 nothing. A zero (or untrained-to-zero) embedding is a clean "opt out" &mdash; the feature
                 simply does not interact &mdash; which is why FM degrades gracefully as features are added or
                 removed. You can confirm with the square-of-sum identity: the new sum is still
                 $[0.7,0.2,0.2]$, and $\sum e_3^2 = 0$, so the halved difference is unchanged.

</details>